# Lab 1 — Additional Experiments
## MSDS 684: Reinforcement Learning — Week 1

This notebook contains experiments beyond those required in the PDF submission.
All experiments use the same `GaussianBanditEnv`, `EpsilonGreedyAgent`, and `UCBAgent`
classes defined in `Lab1_updated.ipynb`.

**Additional experiments:**
1. Greedy baseline (ε = 0) — what happens with zero exploration?
2. Non-stationary bandit — arm means drift over time; constant step-size vs sample-average
3. Cumulative regret curves — formal regret comparison between UCB and ε-greedy
4. Sensitivity to k — how does performance change with 5, 10, 15, 20 arms?


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import gymnasium as gym
import gymnasium.spaces as spaces
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_RUNS  = 1000
N_STEPS = 2000
K       = 10

# ── Taxi version compatibility ────────────────────────────────────────────
_taxi_envs = [e for e in gym.envs.registry.keys() if 'Taxi' in e]
TAXI_ENV = 'Taxi-v4' if 'Taxi-v4' in _taxi_envs else 'Taxi-v3'
print(f'NumPy {np.__version__} | Gymnasium {gym.__version__} | Using {TAXI_ENV}')


## Core Classes (copied from Lab1_updated.ipynb for standalone use)

In [ ]:
class GaussianBanditEnv(gym.Env):
    """10-armed Gaussian bandit following the Gymnasium API."""
    def __init__(self, k: int = 10, reward_std: float = 1.0):
        super().__init__()
        self.k = k
        self.reward_std = reward_std
        self.observation_space = spaces.Discrete(1)
        self.action_space      = spaces.Discrete(k)
        self.q_star = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.q_star      = self.np_random.standard_normal(self.k)
        self.optimal_arm = int(np.argmax(self.q_star))
        return 0, {'optimal_action': self.optimal_arm, 'q_star': self.q_star.copy()}

    def step(self, action):
        reward = self.np_random.normal(self.q_star[action], self.reward_std)
        return 0, reward, False, False, {'optimal_action': self.optimal_arm}


class EpsilonGreedyAgent:
    def __init__(self, k, epsilon, alpha=None):
        """alpha=None uses sample-average; float uses constant step-size."""
        self.k = k
        self.epsilon = epsilon
        self.alpha = alpha
        self.reset()

    def reset(self):
        self.Q = np.zeros(self.k)
        self.N = np.zeros(self.k, dtype=int)

    def select_action(self, rng):
        if rng.random() < self.epsilon:
            return int(rng.integers(self.k))
        return int(np.argmax(self.Q))  # ties -> lowest index

    def update(self, action, reward):
        self.N[action] += 1
        if self.alpha is None:
            lr = 1.0 / self.N[action]  # sample-average
        else:
            lr = self.alpha             # constant step-size
        self.Q[action] += lr * (reward - self.Q[action])


class UCBAgent:
    def __init__(self, k, c):
        self.k = k
        self.c = c
        self.reset()

    def reset(self):
        self.Q = np.zeros(self.k)
        self.N = np.zeros(self.k, dtype=int)
        self.t = 0

    def select_action(self, rng=None):
        unpulled = np.where(self.N == 0)[0]
        if len(unpulled):
            return int(unpulled[0])
        bonus = self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(self.Q + bonus))

    def update(self, action, reward):
        self.t += 1
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


def run_experiment(agent, n_runs=N_RUNS, n_steps=N_STEPS):
    """Run experiment; returns avg_rewards, pct_optimal, sem_rewards, sem_optimal."""
    k = agent.k
    rewards_all = np.zeros((n_runs, n_steps))
    optimal_all = np.zeros((n_runs, n_steps), dtype=bool)

    for run in range(n_runs):
        env = GaussianBanditEnv(k=k)
        env.reset(seed=run)
        agent.reset()
        agent_rng = np.random.default_rng(seed=run + 50_000)

        for t in range(n_steps):
            action = agent.select_action(agent_rng)
            _, reward, _, _, info = env.step(action)
            agent.update(action, reward)
            rewards_all[run, t] = reward
            optimal_all[run, t] = (action == info['optimal_action'])

    avg_r   = rewards_all.mean(axis=0)
    pct_opt = optimal_all.mean(axis=0) * 100.0
    sem_r   = rewards_all.std(axis=0) / np.sqrt(n_runs)
    sem_opt = (optimal_all.std(axis=0) / np.sqrt(n_runs)) * 100.0
    return avg_r, pct_opt, sem_r, sem_opt


print('All classes and run_experiment() defined.')


---
## Experiment 1: Greedy Baseline (ε = 0)

What happens with zero exploration? A pure greedy agent exploits its current best
estimate with no randomness. Because all Q-values start at 0, the first arm pulled
for each slot determines the initial estimates — then greedy locks onto whichever
arm first returned a high reward, regardless of whether it is actually optimal.

**Hypothesis:** Greedy will converge to a suboptimal arm in most runs and achieve
significantly lower % optimal than even ε=0.01.

In [ ]:
steps = np.arange(1, N_STEPS + 1)

# Run greedy (epsilon=0) alongside the three epsilon-greedy configs for comparison
epsilons_ext  = [0.0, 0.01, 0.1, 0.2]
colors_ext    = ['#2c3e50', '#e74c3c', '#3498db', '#2ecc71']
results_greedy = {}

print('Running greedy baseline + ε-greedy configs...')
for eps in epsilons_ext:
    agent = EpsilonGreedyAgent(k=K, epsilon=eps)
    avg_r, pct_opt, sem_r, sem_opt = run_experiment(agent)
    results_greedy[eps] = (avg_r, pct_opt, sem_r, sem_opt)
    print(f'  ε={eps:.2f}  avg reward={avg_r[-1]:.4f}  % optimal={pct_opt[-1]:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Greedy Baseline vs ε-Greedy — 10-Armed Gaussian Bandit\n'
             '1 000 runs · 2 000 steps | shading = ±1 SEM',
             fontsize=13, fontweight='bold', y=1.02)

for eps, color in zip(epsilons_ext, colors_ext):
    avg_r, pct_opt, sem_r, sem_opt = results_greedy[eps]
    lbl = f'Greedy (ε=0)' if eps == 0 else f'ε = {eps}'
    ls  = '--' if eps == 0 else '-'
    ax1.plot(steps, avg_r,   color=color, lw=2.0, ls=ls, label=lbl)
    ax1.fill_between(steps, avg_r-sem_r, avg_r+sem_r, color=color, alpha=0.15)
    ax2.plot(steps, pct_opt, color=color, lw=2.0, ls=ls, label=lbl)
    ax2.fill_between(steps, pct_opt-sem_opt, pct_opt+sem_opt, color=color, alpha=0.15)

ax1.set(xlabel='Time Steps', ylabel='Average Reward',
        title='Average Reward over Time', xlim=(1, N_STEPS))
ax1.legend(fontsize=10)
ax2.set(xlabel='Time Steps', ylabel='% Optimal Action',
        title='Optimal Action % over Time', xlim=(1, N_STEPS), ylim=(0, 100))
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('visualizations/greedy_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

greedy_pct = results_greedy[0.0][1][-1]
print(f'\nGreedy (ε=0) final % optimal: {greedy_pct:.1f}%')
print(f'This confirms greedy locks onto a suboptimal arm in most runs —')
print(f'it cannot recover without any exploration.')


---
## Experiment 2: Non-Stationary Bandit

In the standard bandit above, arm means are fixed for the entire episode. Real-world
problems are often **non-stationary** — reward distributions drift over time.

Here we add Gaussian random walks to the arm means at every step:
q*(a) <- q*(a) + N(0, 0.01) for all arms at each time step.

**Sample-average** weights all past observations equally, making it slow to adapt.
**Constant step-size** (α = 0.1) weights recent rewards more heavily, tracking drift.

**Hypothesis:** Constant step-size will significantly outperform sample-average
on the non-stationary bandit (S&B Section 2.5).

In [ ]:
class NonStationaryBanditEnv(gym.Env):
    """10-armed bandit where arm means drift by N(0, drift_std) each step."""
    def __init__(self, k=10, reward_std=1.0, drift_std=0.01):
        super().__init__()
        self.k          = k
        self.reward_std = reward_std
        self.drift_std  = drift_std
        self.observation_space = spaces.Discrete(1)
        self.action_space      = spaces.Discrete(k)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.q_star      = self.np_random.standard_normal(self.k)
        self.optimal_arm = int(np.argmax(self.q_star))
        return 0, {'optimal_action': self.optimal_arm}

    def step(self, action):
        # Drift all arm means
        self.q_star      += self.np_random.normal(0, self.drift_std, self.k)
        self.optimal_arm  = int(np.argmax(self.q_star))
        reward = self.np_random.normal(self.q_star[action], self.reward_std)
        return 0, reward, False, False, {'optimal_action': self.optimal_arm}


def run_nonstationary(agent_cls, agent_kwargs, n_runs=N_RUNS, n_steps=N_STEPS):
    """Same structure as run_experiment but uses NonStationaryBanditEnv."""
    rewards_all = np.zeros((n_runs, n_steps))
    optimal_all = np.zeros((n_runs, n_steps), dtype=bool)

    for run in range(n_runs):
        env   = NonStationaryBanditEnv(k=K)
        env.reset(seed=run)
        agent = agent_cls(**agent_kwargs)
        agent_rng = np.random.default_rng(seed=run + 50_000)

        for t in range(n_steps):
            action = agent.select_action(agent_rng)
            _, reward, _, _, info = env.step(action)
            agent.update(action, reward)
            rewards_all[run, t] = reward
            optimal_all[run, t] = (action == info['optimal_action'])

    avg_r   = rewards_all.mean(axis=0)
    pct_opt = optimal_all.mean(axis=0) * 100.0
    sem_r   = rewards_all.std(axis=0) / np.sqrt(n_runs)
    sem_opt = (optimal_all.std(axis=0) / np.sqrt(n_runs)) * 100.0
    return avg_r, pct_opt, sem_r, sem_opt


print('Running non-stationary experiment...')
configs_ns = [
    ('Sample-avg  ε=0.1  α=1/n', '#e74c3c',
     EpsilonGreedyAgent, {'k': K, 'epsilon': 0.1, 'alpha': None}),
    ('Const α=0.1 ε=0.1',        '#3498db',
     EpsilonGreedyAgent, {'k': K, 'epsilon': 0.1, 'alpha': 0.1}),
    ('Const α=0.1 ε=0.01',       '#2ecc71',
     EpsilonGreedyAgent, {'k': K, 'epsilon': 0.01, 'alpha': 0.1}),
]
results_ns = {}
for label, color, cls, kwargs in configs_ns:
    avg_r, pct_opt, sem_r, sem_opt = run_nonstationary(cls, kwargs)
    results_ns[label] = (avg_r, pct_opt, sem_r, sem_opt, color)
    print(f'  {label:<28}  reward={avg_r[-1]:.4f}  % optimal={pct_opt[-1]:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Non-Stationary Bandit: Sample-Average vs Constant Step-Size\n'
             '1 000 runs · 2 000 steps | drift N(0, 0.01) per step | shading = ±1 SEM',
             fontsize=12, fontweight='bold', y=1.02)

for label, (avg_r, pct_opt, sem_r, sem_opt, color) in results_ns.items():
    ax1.plot(steps, avg_r,   color=color, lw=2.0, label=label)
    ax1.fill_between(steps, avg_r-sem_r, avg_r+sem_r, color=color, alpha=0.15)
    ax2.plot(steps, pct_opt, color=color, lw=2.0, label=label)
    ax2.fill_between(steps, pct_opt-sem_opt, pct_opt+sem_opt, color=color, alpha=0.15)

ax1.set(xlabel='Time Steps', ylabel='Average Reward',
        title='Average Reward (Non-Stationary)', xlim=(1, N_STEPS))
ax1.legend(fontsize=9)
ax2.set(xlabel='Time Steps', ylabel='% Optimal Action',
        title='Optimal Action % (Non-Stationary)', xlim=(1, N_STEPS), ylim=(0, 100))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/nonstationary_bandit.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey finding: constant step-size tracks the drifting optimum;')
print('sample-average converges to stale estimates and falls behind (S&B Section 2.5).')


---
## Experiment 3: Cumulative Regret Curves

Average reward curves show performance at each step, but **cumulative regret**
shows the total cost of suboptimal choices over time:

Regret(T) = sum_{t=1}^{T} [q*(a*) - q*(a_t)]

where a* is the optimal arm and a_t is the chosen arm at step t.
Lower regret = better. UCB1 theory guarantees O(ln T) regret; fixed ε-greedy is O(T).
This experiment visualizes that gap directly.

In [ ]:
def run_regret_experiment(agent, n_runs=500, n_steps=N_STEPS):
    """Returns avg cumulative regret and SEM across runs."""
    k = agent.k
    regret_all = np.zeros((n_runs, n_steps))

    for run in range(n_runs):
        env = GaussianBanditEnv(k=k)
        _, info = env.reset(seed=run)
        agent.reset()
        agent_rng = np.random.default_rng(seed=run + 50_000)
        cumulative_regret = 0.0
        q_star = info['q_star']
        best_q = q_star.max()

        for t in range(n_steps):
            action = agent.select_action(agent_rng)
            _, _, _, _, step_info = env.step(action)
            # Regret = gap between optimal mean and chosen arm mean
            cumulative_regret += best_q - q_star[action]
            regret_all[run, t]  = cumulative_regret
            agent.update(action, env.q_star[action] +
                         np.random.default_rng(run+t).normal(0, 1))  # noisy

    return regret_all.mean(axis=0), regret_all.std(axis=0) / np.sqrt(n_runs)


print('Running regret experiment (500 runs for speed)...')
regret_configs = [
    ('ε-Greedy ε=0.01', '#e74c3c', EpsilonGreedyAgent(k=K, epsilon=0.01)),
    ('ε-Greedy ε=0.10', '#3498db', EpsilonGreedyAgent(k=K, epsilon=0.10)),
    ('UCB c=0.5',        '#e67e22', UCBAgent(k=K, c=0.5)),
    ('UCB c=1.0',        '#8e44ad', UCBAgent(k=K, c=1.0)),
    ('UCB c=2.0',        '#16a085', UCBAgent(k=K, c=2.0)),
]
regret_results = {}
for label, color, agent in regret_configs:
    avg_reg, sem_reg = run_regret_experiment(agent, n_runs=500)
    regret_results[label] = (avg_reg, sem_reg, color)
    print(f'  {label:<20}  final regret = {avg_reg[-1]:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Cumulative Regret — ε-Greedy vs UCB\n'
             '500 runs · 2 000 steps | lower is better',
             fontsize=13, fontweight='bold', y=1.02)

for label, (avg_reg, sem_reg, color) in regret_results.items():
    ax.plot(steps, avg_reg, color=color, lw=2.0, label=label)
    ax.fill_between(steps, avg_reg-sem_reg, avg_reg+sem_reg, color=color, alpha=0.15)

ax.set(xlabel='Time Steps', ylabel='Cumulative Regret',
       title='Cumulative Regret over Time', xlim=(1, N_STEPS))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('visualizations/regret_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nUCB regret grows sub-linearly (concave curve) — consistent with O(ln T) theory.')
print('ε-greedy regret grows linearly — O(T) as predicted.')


---
## Experiment 4: Sensitivity to Number of Arms (k)

How does algorithm performance change as the number of arms increases from 5 to 20?
With more arms, finding the optimal one is harder — both algorithms should show
degraded % optimal, but UCB may degrade more gracefully because its exploration
budget scales with uncertainty rather than being fixed at ε.

In [ ]:
k_values = [5, 10, 15, 20]
eps_k_results = {}  # (k, eps) -> final % optimal
ucb_k_results = {}  # (k, c)   -> final % optimal

print('Running k-sensitivity experiment...')
for k in k_values:
    for eps in [0.01, 0.1]:
        agent = EpsilonGreedyAgent(k=k, epsilon=eps)
        _, pct_opt, _, _ = run_experiment(agent, n_runs=500, n_steps=N_STEPS)
        eps_k_results[(k, eps)] = pct_opt[-1]
    for c in [0.5, 1.0]:
        agent = UCBAgent(k=k, c=c)
        _, pct_opt, _, _ = run_experiment(agent, n_runs=500, n_steps=N_STEPS)
        ucb_k_results[(k, c)] = pct_opt[-1]
    print(f'  k={k} done')

fig, ax = plt.subplots(figsize=(9, 5))
eps_styles = {0.01: ('o-', '#e74c3c'), 0.1: ('s-', '#3498db')}
ucb_styles = {0.5:  ('^-', '#e67e22'), 1.0: ('D-', '#8e44ad')}

for eps, (marker, color) in eps_styles.items():
    vals = [eps_k_results[(k, eps)] for k in k_values]
    ax.plot(k_values, vals, marker, color=color, lw=2.0,
            label=f'ε-Greedy ε={eps}', ms=8)

for c, (marker, color) in ucb_styles.items():
    vals = [ucb_k_results[(k, c)] for k in k_values]
    ax.plot(k_values, vals, marker, color=color, lw=2.0,
            label=f'UCB c={c}', ms=8)

ax.set(xlabel='Number of Arms (k)', ylabel='% Optimal Action at Step 2000',
       title='Performance vs Number of Arms\n(500 runs, step 2000)',
       xticks=k_values)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/k_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSummary:')
print(f'{"Config":<22} {"k=5":>6} {"k=10":>6} {"k=15":>6} {"k=20":>6}')
print('-' * 46)
for eps, (_, _) in eps_styles.items():
    vals = [f"{eps_k_results[(k,eps)]:.1f}%" for k in k_values]
    print(f'  ε-Greedy ε={eps:<5}  {vals[0]:>6} {vals[1]:>6} {vals[2]:>6} {vals[3]:>6}')
for c, (_, _) in ucb_styles.items():
    vals = [f"{ucb_k_results[(k,c)]:.1f}%" for k in k_values]
    print(f'  UCB c={c:<7}    {vals[0]:>6} {vals[1]:>6} {vals[2]:>6} {vals[3]:>6}')


---
## Summary of Additional Findings

| Experiment | Key Finding | S&B Reference |
|---|---|---|
| Greedy (ε=0) | Locks onto suboptimal arm in most runs; near-zero % optimal | Ch. 2.2 |
| Non-stationary | Constant step-size (α=0.1) tracks drift; sample-average falls behind | Section 2.5 |
| Regret curves | UCB regret is concave (O(ln T)); ε-greedy is linear (O(T)) | Section 2.7 |
| k sensitivity | All algorithms degrade with more arms; UCB degrades more gracefully | Section 2.7 |

These experiments reinforce that algorithm choice depends on the problem horizon,
stationarity, and number of actions — themes that extend directly to full MDPs
in Weeks 2–5.
